# Node Import Chain and Side Effects

> **Safety boundary:** This workbook performs static analysis of sanitized excerpts, fake fixtures, and derived public metadata. It does not execute attacker code, inspect the analyst's environment, or contact any network.

## Static Call Chain

```text
server.js
  -> require("./config")
  -> require("./socket/index")
       -> require("../controllers/auth")
       -> validateApiKey() at module top level
            -> setApiKey(...)
            -> verify(...)
            -> compile response as code with CommonJS access
            -> invoke compiled code with require
```

CommonJS evaluates a module's top-level statements the first time it is
required. A server import can therefore trigger network and execution behavior
before any route, socket event, or user interaction.

## Sanitized Source Map

The following records preserve relationships and line-level meaning without
embedding executable source:

| File | Static relationship | Side effect |
|---|---|---|
| `server.js` | imports config, then socket module | causes imported modules to evaluate |
| `config.js` | initializes dotenv-style configuration | populates process environment |
| `socket/index.js` | imports auth controller | invokes validation at module top level |
| `controllers/auth.js` | exports validation helpers | posts environment and grants returned text Node authority |

In [ ]:
files = {
    "server.js": [
        {"kind": "require", "target": "./config"},
        {"kind": "require", "target": "./socket/index"},
    ],
    "socket/index.js": [
        {"kind": "require", "target": "../controllers/auth"},
        {"kind": "top_level_call", "target": "validateApiKey"},
    ],
    "controllers/auth.js": [
        {"kind": "function_call", "target": "setApiKey"},
        {"kind": "function_call", "target": "verify"},
        {"kind": "authority_transfer", "target": "response text receives CommonJS module access"},
    ],
}

edges = [
    {"source": source, "target": item["target"], "kind": item["kind"]}
    for source, items in files.items()
    for item in items
]
edges

In [ ]:
top_level_effects = [
    {
        "module": "config.js",
        "trigger": "first import",
        "effect": "configuration loader mutates process environment",
        "requires_user_request": False,
    },
    {
        "module": "socket/index.js",
        "trigger": "first import",
        "effect": "validation routine begins",
        "requires_user_request": False,
    },
]
top_level_effects

## Dangerous Primitive Table

| Primitive | Role in the path | Security consequence |
|---|---|---|
| top-level `validateApiKey()` | begins behavior during import | startup becomes detonation |
| bulk environment copy | creates request body | broad secret exposure |
| response text compiled as code | turns data into instructions | remote stage delivery |
| `require` passed to downloaded code | grants Node module authority | filesystem, OS, and network capability |

The exact hostile syntax is documented in the public report as
`new Function("require", response.data)` followed by invocation with `require`.
It is shown here only as inert Markdown and is never run.

In [ ]:
review_questions = [
    {"primitive": "module import", "question": "Does top-level evaluation call a function?"},
    {"primitive": "HTTP response", "question": "Is response content treated as data or instructions?"},
    {"primitive": "module capability", "question": "Is a loader handed require or equivalent authority?"},
    {"primitive": "async validation", "question": "Is the caller actually awaiting the result?"},
]
review_questions

## Review Lesson

A filename such as `socket/index.js` suggests infrastructure glue, but the
review unit is the import graph plus top-level statements. The trust boundary
collapsed because loading a module also began validation, transmission, stage
retrieval, and dynamic execution.

**Observed:** runtime evidence places the downloaded stage under the executor
originating from `socket/index.js`.

**Not proven by static imports:** which request bytes reached the server or
which later tasking ran.

**Detection:** grep for top-level calls near imports; build AST rules for
response-to-code flows; flag a function constructor receiving response data;
review imports whose initialization performs network I/O.

**Confidence upgrade:** Node module-load tracing, process telemetry, proxy logs,
or a preserved server-side request record.